In [1]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

In [2]:
import pathlib
import torch
import torch.nn as nn
from tqdm import tqdm
from torchmetrics.regression import (
    R2Score,
    MeanSquaredError,
    MeanAbsoluteError,
    PearsonCorrCoef,
)

from egnn.qm9.models import EGNN
from egnn.qm9 import utils as qm9_utils
from src.dataset import EGNNQM9DataModule

torch.set_float32_matmul_precision("high")

/home/pham/miniforge3/envs/cp3d/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class ARGS:
    def __init__(self, configs):
        for key, value in configs.items():
            setattr(self, key, value)


configs = {
    "data_dir": pathlib.Path("data/egnn/QM9"),
    "seed": 43,
    "amp": True,
    "log_dir": pathlib.Path("./results/egnn/qm9"),
    "epochs": 15,
    "eval_interval": 2,
    "save_ckpt_path": pathlib.Path("./results/egnn/qm9/new_env_test_egnn.pth"),
    "learning_rate": 0.002,
    "min_learning_rate": 1e-7,
    "batch_size": 1024,
    "nf": 128,
    "n_layers": 7,
    "attention": 1,
    "node_attr": 0,
    "property": "homo",
    "charge_power": 2,
}

args = ARGS(configs)

In [4]:
model = EGNN(
    in_node_nf=15,
    in_edge_nf=0,
    hidden_nf=args.nf,
    n_layers=args.n_layers,
    coords_weight=1.0,
    attention=args.attention,
    node_attr=args.node_attr,
)

In [5]:
loss_fn = nn.L1Loss()
datamodule = EGNNQM9DataModule(
    data_dir=args.data_dir,
    batch_size=args.batch_size,
)
train_dataloader = datamodule.train_dataloader()
val_dataloader = datamodule.val_dataloader()

In [6]:
values = train_dataloader.dataset.data[args.property]
mean = torch.mean(values)
mad = torch.mean(torch.abs(values - mean))

In [7]:
device = torch.cuda.current_device()
model.to(device=device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=args.learning_rate,
    fused=True,
)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, args.epochs)

mae = MeanAbsoluteError()
rmse = MeanSquaredError(squared=False)
r2 = R2Score()
pearson = PearsonCorrCoef()

for epoch in range(args.epochs):
    loss_acc = torch.zeros((1,), device=device)
    model.train()
    for i, batch in tqdm(
        enumerate(train_dataloader),
        total=len(train_dataloader),
        unit="batch",
        desc=f"Epoch {epoch + 1}",
    ):
        batch_size, n_nodes, _ = batch["positions"].size()
        atom_positions = (
            batch["positions"]
            .view(batch_size * n_nodes, -1)
            .to(device, torch.float32, non_blocking=True)
        )
        atom_mask = (
            batch["atom_mask"]
            .view(batch_size * n_nodes, -1)
            .to(device, torch.float32, non_blocking=True)
        )
        edge_mask = batch["edge_mask"].to(device, torch.float32, non_blocking=True)
        one_hot = batch["one_hot"].to(device, torch.float32, non_blocking=True)
        charges = batch["charges"].to(device, torch.float32, non_blocking=True)
        nodes = qm9_utils.preprocess_input(
            one_hot, charges, args.charge_power, datamodule.max_charge, device
        )

        nodes = nodes.view(batch_size * n_nodes, -1)
        edges = qm9_utils.get_adj_matrix(n_nodes, batch_size, device)
        target = batch[args.property].to(device, torch.float32, non_blocking=True)

        pred = model(
            h0=nodes,
            x=atom_positions,
            edges=edges,
            edge_attr=None,
            node_mask=atom_mask,
            edge_mask=edge_mask,
            n_nodes=n_nodes,
        )
        loss = loss_fn(pred.flatten(), (target.flatten() - mean) / mad)
        loss_acc += loss.detach()
        loss.backward()
        optimizer.step()
        model.zero_grad(set_to_none=True)
        # lr_scheduler.step()
    print(f"Epoch {epoch + 1}")
    print(f"Loss {loss_acc.item() / len(train_dataloader):.4f}")

    model.eval()
    with torch.inference_mode():
        for i, batch in tqdm(
            enumerate(val_dataloader),
            total=len(val_dataloader),
            unit="batch",
            desc=f"Epoch {epoch + 1}",
            leave=False,
        ):
            batch_size, n_nodes, _ = batch["positions"].size()
        atom_positions = (
            batch["positions"]
            .view(batch_size * n_nodes, -1)
            .to(device, torch.float32, non_blocking=True)
        )
        atom_mask = (
            batch["atom_mask"]
            .view(batch_size * n_nodes, -1)
            .to(device, torch.float32, non_blocking=True)
        )
        edge_mask = batch["edge_mask"].to(device, torch.float32, non_blocking=True)
        one_hot = batch["one_hot"].to(device, torch.float32, non_blocking=True)
        charges = batch["charges"].to(device, torch.float32, non_blocking=True)
        nodes = qm9_utils.preprocess_input(
            one_hot, charges, args.charge_power, datamodule.max_charge, device
        )

        nodes = nodes.view(batch_size * n_nodes, -1)
        edges = qm9_utils.get_adj_matrix(n_nodes, batch_size, device)
        target = batch[args.property].to(device, torch.float32, non_blocking=True)

        pred = model(
            h0=nodes,
            x=atom_positions,
            edges=edges,
            edge_attr=None,
            node_mask=atom_mask,
            edge_mask=edge_mask,
            n_nodes=n_nodes,
        )
        pred = mad * pred + mean
        pred_flat, target_flat = (
            pred.detach().view(-1).float().cpu(),
            target.detach().view(-1).float().cpu(),
        )

        mae(pred_flat, target_flat)
        rmse(pred_flat, target_flat)
        r2(pred_flat, target_flat)
        pearson(pred_flat, target_flat)

        print(
            f"""
            validation MAE: {float(mae.compute()):.4f}
            validation RMSE: {float(rmse.compute()):.4f}
            validation R²: {float(r2.compute()):.4f}
            validation Pearson r: {float(pearson.compute()):.4f}
            """
        )

        mae.reset()
        rmse.reset()
        r2.reset()
        pearson.reset()

Epoch 1: 100%|██████████| 98/98 [00:19<00:00,  5.11batch/s]

Epoch 1
Loss 1.0156



            validation MAE: 0.0149
            validation RMSE: 0.0205
            validation R²: 0.1231
            validation Pearson r: 0.3527
            


Epoch 2: 100%|██████████| 98/98 [00:18<00:00,  5.26batch/s]


Epoch 2
Loss 0.9190


...

Epoch 14
Loss 0.3102



            validation MAE: 0.0046
            validation RMSE: 0.0064
            validation R²: 0.9152
            validation Pearson r: 0.9586
            


Epoch 15: 100%|██████████| 98/98 [00:18<00:00,  5.29batch/s]


Epoch 15
Loss 0.2892



            validation MAE: 0.0042
            validation RMSE: 0.0057
            validation R²: 0.9319
            validation Pearson r: 0.9663
            
